# NirmalLLM — Fine-Tuned Medical Chat Demo

**What this notebook does**

This notebook presents the completed fine-tuning project:
- Explains *how* fine-tuning was done (method, data, template, LoRA config)
- Shows the evaluation metrics from the final training run
- Displays side-by-side generation comparisons: **Base Qwen3-1.7B** vs **NirmalLLM** (fine-tuned)
- Explains what each metric means and what the numbers tell us

The generation comparisons are pre-recorded outputs from the GPU session (Kaggle T4).  
This notebook runs on CPU with no GPU or ML framework required.

**Architecture refs:** ARCHITECTURE.md A, C, D, E1–E6; IMPLEMENTATION_PLAN Stage 3–5.

---

## 0. Parameters

In [1]:
# ── Parameters ────────────────────────────────────────────────────────────────
import os
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'

_p = Path().resolve()
REPO_ROOT = _p
for _ in range(6):
    if (_p / 'sft').is_dir():
        REPO_ROOT = _p
        break
    _p = _p.parent

SFT_DIR     = REPO_ROOT / 'sft'
WORK_DIR    = SFT_DIR / 'work'
ADAPTER_DIR = WORK_DIR / 'adapter'
RESULTS_DIR = REPO_ROOT / 'sft_results_unzipped'

print('REPO_ROOT   :', REPO_ROOT)
print('ADAPTER_DIR :', ADAPTER_DIR)
print('adapter exists:', ADAPTER_DIR.exists())

REPO_ROOT   : C:\Users\nirmi\Desktop\Capstone
ADAPTER_DIR : C:\Users\nirmi\Desktop\Capstone\sft\work\adapter
adapter exists: True


---
## 1. Definitions

In [2]:
# ── Definitions ───────────────────────────────────────────────────────────────
import json, math

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def nll_to_ppl(nll):
    """Convert negative log-likelihood (nats) to perplexity."""
    return round(math.exp(nll), 4)

def improvement_pct(base, finetuned):
    """Percentage reduction from base to finetuned."""
    return round((base - finetuned) / base * 100, 1)

print('definitions loaded')

definitions loaded


---
## 2. In-notebook tests (must pass before any data is read)

In [3]:
# ── Tests ─────────────────────────────────────────────────────────────────────
# Synthetic fixtures only — no dataset text.

# T1: nll_to_ppl is inverse of log
assert abs(nll_to_ppl(0.0) - 1.0) < 1e-6, 'T1 fail: ppl(0)!=1'
assert abs(nll_to_ppl(math.log(2)) - 2.0) < 1e-4, 'T1 fail: ppl(log2)!=2'

# T2: improvement_pct correct
assert improvement_pct(2.0, 1.6) == 20.0, 'T2 fail'
assert improvement_pct(1.0, 1.0) == 0.0,  'T2 fail zero'

# T3: load_json works on synthetic dict
import tempfile
_tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False, encoding='utf-8')
_tmp.write(json.dumps({'x': 1}))
_tmp.close()
assert load_json(_tmp.name) == {'x': 1}, 'T3 fail'
Path(_tmp.name).unlink()

# T4: required files exist
assert ADAPTER_DIR.exists(), f'T4 fail: adapter dir missing at {ADAPTER_DIR}'
assert (ADAPTER_DIR / 'adapter_config.json').exists(), 'T4 fail: adapter_config.json missing'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), 'T4 fail: adapter_model.safetensors missing'
assert (RESULTS_DIR / 'run_manifest.json').exists(), 'T4 fail: run_manifest.json missing'
assert (RESULTS_DIR / 'val_metrics.json').exists(), 'T4 fail: val_metrics.json missing'

print('ALL TESTS PASSED (T1–T4)')

ALL TESTS PASSED (T1–T4)


---
## 3. How fine-tuning was done

### 3.1 The base model

| Property | Value |
|---|---|
| Model | `Qwen/Qwen3-1.7B` |
| Pinned revision | `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e` |
| Parameters | 1.7B total (1.4B non-embedding) |
| Architecture | 28 layers, grouped-query attention (16/8 heads) |
| License | Apache-2.0 |

Qwen3-1.7B is a post-trained (instruction-following) model. We replace its default chat template — which toggles a "thinking" mode — with our own clean ChatML template that has no `<think>` blocks.

### 3.2 The method: LoRA SFT

**LoRA (Low-Rank Adaptation)** freezes the base model weights and injects small trainable rank-decomposition matrices into every linear layer. Instead of updating 1.7 billion parameters, we train only ~17 million (~1%).

```
Original weight matrix W  (frozen)
        ↓
W + BA   where B ∈ R^(d×r), A ∈ R^(r×k), r=16  (trainable)
```

We used **plain HuggingFace Trainer + PEFT**, not TRL, Unsloth, QLoRA or DeepSpeed. This gives full control over the loss mask (see §3.4).

### 3.3 The data

| Source | Dialogues | Character |
|---|---|---|
| MedDialog | ~112k | Real doctor–patient exchanges, single-exchange only, formal clinical style |
| HealthChat-WildChat | ~11k | Multi-turn health Q&A, LLM-written assistant side |

Training used **15,625 dialogues** sampled from the full pool (25k budget; actual JSONL contained 15,625 eligible after filtering). Every dialogue is a complete conversation serialised as a single ChatML sequence.

### 3.4 The loss mask

Only **assistant tokens** contribute to the loss. User turns, system prompt, role headers, and padding are all masked to `-100` (ignored by cross-entropy).

```
<|im_start|>system\n...system prompt...<|im_end|>\n   → labels = -100 (masked)
<|im_start|>user\n...question...<|im_end|>\n           → labels = -100 (masked)
<|im_start|>assistant\n                                → labels = -100 (masked)
...answer...<|im_end|>                                 → labels = token ids  ✓ SUPERVISED
\n                                                     → labels = -100 (masked)
```

This means the model is only rewarded for producing good *assistant* responses — it never "wastes" capacity memorising the questions.

### 3.5 The ChatML template and system prompt

Every sequence starts with a fixed system prompt (hash-pinned so it cannot silently drift):

> *You are a general health-information assistant. You are an AI, not a doctor: you cannot examine the user, diagnose, or prescribe. Use everything the user has told you in this conversation. If key details are missing, ask one or two focused questions. State uncertainty plainly. If there are signs of an emergency, tell the user to contact local emergency services now. Recommend seeing a qualified clinician for severe, persistent, or worsening symptoms, and for any prescription medicine or dosing decision.*

In [4]:
# ── Print LoRA config from the saved adapter ──────────────────────────────────
adapter_cfg = load_json(ADAPTER_DIR / 'adapter_config.json')

lora_summary = {
    'peft_type'    : adapter_cfg['peft_type'],
    'r'            : adapter_cfg['r'],
    'lora_alpha'   : adapter_cfg['lora_alpha'],
    'lora_dropout' : adapter_cfg['lora_dropout'],
    'target_modules': sorted(adapter_cfg['target_modules']),
    'use_dora'     : adapter_cfg['use_dora'],
    'use_rslora'   : adapter_cfg['use_rslora'],
}

print('=== LoRA Adapter Configuration ===')
for k, v in lora_summary.items():
    print(f'  {k:<20}: {v}')

=== LoRA Adapter Configuration ===
  peft_type           : LORA
  r                   : 16
  lora_alpha          : 32
  lora_dropout        : 0.05
  target_modules      : ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
  use_dora            : False
  use_rslora          : False


In [5]:
# ── Print training run configuration ─────────────────────────────────────────
manifest = load_json(RESULTS_DIR / 'run_manifest.json')
cfg = manifest['config']

print('=== Training Run Configuration ===')
train_params = [
    ('base_model',       'Qwen3-1.7B'),
    ('model_revision',   manifest['model_revision']),
    ('gpu',              manifest['gpu_probe']['device']),
    ('precision',        manifest['gpu_probe']['training_dtype']),
    ('lr',               cfg['lr']),
    ('epochs',           cfg['epochs']),
    ('effective_batch',  cfg['effective_batch']),
    ('micro_batch',      cfg['micro_batch']),
    ('grad_accum',       cfg['grad_accum']),
    ('max_len',          cfg['max_len']),
    ('train_dialogues',  manifest['train_dialogues']),
    ('max_steps',        cfg['max_steps']),
    ('lora_r',           cfg['lora_r']),
    ('lora_alpha',       cfg['lora_alpha']),
    ('lora_dropout',     cfg['lora_dropout']),
    ('seed',             cfg['seed']),
    ('transformers',     manifest['library_versions']['transformers']),
    ('peft',             manifest['library_versions']['peft']),
    ('torch',            manifest['library_versions']['torch']),
]
for k, v in train_params:
    print(f'  {k:<22}: {v}')

=== Training Run Configuration ===
  base_model            : Qwen3-1.7B
  model_revision        : 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
  gpu                   : Tesla T4
  precision             : fp16
  lr                    : 0.0002
  epochs                : 1
  effective_batch       : 32
  micro_batch           : 1
  grad_accum            : 32
  max_len               : 2048
  train_dialogues       : 15625
  max_steps             : -1
  lora_r                : 16
  lora_alpha            : 32
  lora_dropout          : 0.05
  seed                  : 42
  transformers          : 5.20.2
  peft                  : 0.21.2
  torch                 : 2.11.0+cu130


---
## 4. Evaluation metrics — what they mean

### 4.1 NLL (Negative Log-Likelihood)

NLL measures how well the model predicts the actual assistant tokens in held-out conversations. For each assistant token, the model assigns a probability — NLL is the average negative log of those probabilities.

- **Lower is better.** A perfect model predicting every token with certainty has NLL = 0.
- **What it is not:** NLL measures fit to the training distribution (style, phrasing, structure), not medical correctness or safety. A model could have low NLL and still give wrong advice.
- **Why we measure it per source:** MedDialog (formal clinical Q&A) and HealthChat (conversational) have very different styles, so their NLLs are not directly comparable.

### 4.2 Perplexity

Perplexity = exp(NLL). It is easier to interpret: a perplexity of 5 means the model is as uncertain as if it were choosing uniformly among 5 tokens at every position. Lower is better.

### 4.3 Macro val loss

Simple mean of the two per-source NLLs (MedDialog NLL + HealthChat NLL) / 2. Used as the checkpoint selection criterion — the best checkpoint is the one with the lowest macro val loss, preventing the model from over-optimising for the larger source.

### 4.4 Bootstrap 95% CI

We resample the 1,020 validation dialogues 1,000 times to get a confidence interval on the overall NLL. This tells us how stable the number is — a narrow CI means the result is reliable, not driven by a few outlier dialogues.

### 4.5 First vs later exchange NLL

For multi-turn HealthChat dialogues, we separately measure NLL on the *first* assistant reply versus replies in *later* turns. If fine-tuning learned to use conversation history, later turns should have *lower* NLL than first turns (the model can condition on more context). This is the signal for history utilisation.

In [6]:
# ── Load and display final val metrics ───────────────────────────────────────
vm = load_json(RESULTS_DIR / 'val_metrics.json')

overall      = vm['overall']
nll_src      = vm['nll_per_source']
nll_exch     = vm['nll_by_exchange_position']
macro        = vm['macro_val_loss']
ci           = overall['bootstrap_95ci_nll']

print('══════════════════════════════════════════════')
print('  NirmalLLM — Final Validation Metrics')
print('══════════════════════════════════════════════')
print(f"  Overall NLL          : {overall['nll']:.4f}")
print(f"  Overall Perplexity   : {overall['perplexity']:.4f}")
print(f"  Bootstrap 95% CI NLL : [{ci['lo']:.4f}, {ci['hi']:.4f}]  (n={ci['n_resamples']} resamples)")
print(f"  Macro val loss       : {macro:.4f}")
print(f"  Val dialogues        : {overall['n_dialogues']}")
print(f"  Supervised tokens    : {overall['n_sup_tokens']:,}")
print()
print('  ── Per source ──────────────────────────────')
for src, v in nll_src.items():
    ppl = nll_to_ppl(v['nll'])
    print(f"  {src}")
    print(f"    NLL={v['nll']:.4f}  PPL={ppl}  dialogues={v['n_dialogues']}  sup_tokens={v['n_sup_tokens']:,}")
print()
print('  ── History utilisation (HealthChat multi-turn) ─')
print(f"  First-exchange NLL   : {nll_exch['first_exchange_nll']:.4f}  (n={nll_exch['n_dialogues_first']})")
print(f"  Later-exchange NLL   : {nll_exch['later_exchange_nll']:.4f}  (n={nll_exch['n_dialogues_later']})")
gap = nll_exch['first_exchange_nll'] - nll_exch['later_exchange_nll']
print(f"  Gap (first − later)  : {gap:.4f}  {'↓ model benefits from history' if gap > 0 else '↑ no clear history benefit'}")
print('══════════════════════════════════════════════')

══════════════════════════════════════════════
  NirmalLLM — Final Validation Metrics
══════════════════════════════════════════════
  Overall NLL          : 1.6515
  Overall Perplexity   : 5.2148
  Bootstrap 95% CI NLL : [1.5995, 1.7061]  (n=1000 resamples)
  Macro val loss       : 1.7248
  Val dialogues        : 1020
  Supervised tokens    : 411,244

  ── Per source ──────────────────────────────
  HealthChat-WildChat
    NLL=1.0878  PPL=2.9678  dialogues=223  sup_tokens=180,495
  MedDialog
    NLL=2.3619  PPL=10.6106  dialogues=500  sup_tokens=60,371

  ── History utilisation (HealthChat multi-turn) ─
  First-exchange NLL   : 1.1287  (n=185)
  Later-exchange NLL   : 1.0831  (n=185)
  Gap (first − later)  : 0.0456  ↓ model benefits from history
══════════════════════════════════════════════


---
## 5. Progress across experiments (E0 → E2 → Final)

Every run measured NLL on the same 1,020 held-out dialogues. This table tracks how the model improved.

In [7]:
# ── Experiment progression summary ───────────────────────────────────────────
# Values recorded from run results (counts, NLL numbers only — no dataset text).
experiments = [
    {'run': 'E0  (50 steps,  lr=1e-4,  mb=1)', 'overall_nll': 1.8135, 'meddialog_nll': 2.6478, 'healthchat_nll': 1.1453, 'macro': 1.8966},
    {'run': 'E2a (150 steps, lr=5e-5,  mb=1)', 'overall_nll': 1.7742, 'meddialog_nll': 2.6066, 'healthchat_nll': 1.0986, 'macro': 1.8526},
    {'run': 'E2b (150 steps, lr=1e-4,  mb=1)', 'overall_nll': 1.8135, 'meddialog_nll': 2.6478, 'healthchat_nll': 1.1453, 'macro': 1.8966},
    {'run': 'E2c (150 steps, lr=2e-4,  mb=1)', 'overall_nll': 1.7132, 'meddialog_nll': 2.4957, 'healthchat_nll': 1.0776, 'macro': 1.7867},
    {'run': 'FINAL (full epoch, lr=2e-4, mb=1)', 'overall_nll': 1.6515, 'meddialog_nll': 2.3619, 'healthchat_nll': 1.0878, 'macro': 1.7248},
]

print(f"{'Run':<42} {'Overall NLL':>12} {'PPL':>7} {'MedDialog NLL':>14} {'HealthChat NLL':>15} {'Macro':>7}")
print('-' * 101)
for e in experiments:
    ppl = nll_to_ppl(e['overall_nll'])
    marker = ' ◀ best' if e['run'].startswith('FINAL') else ''
    print(f"{e['run']:<42} {e['overall_nll']:>12.4f} {ppl:>7.3f} {e['meddialog_nll']:>14.4f} {e['healthchat_nll']:>15.4f} {e['macro']:>7.4f}{marker}")

base_nll  = experiments[0]['overall_nll']
final_nll = experiments[-1]['overall_nll']
print()
print(f'Overall NLL improvement E0 → Final : {improvement_pct(base_nll, final_nll)}%')
print(f'Perplexity  E0 → Final             : {nll_to_ppl(base_nll)} → {nll_to_ppl(final_nll)}')

Run                                         Overall NLL     PPL  MedDialog NLL  HealthChat NLL   Macro
-----------------------------------------------------------------------------------------------------
E0  (50 steps,  lr=1e-4,  mb=1)                  1.8135   6.132         2.6478          1.1453  1.8966
E2a (150 steps, lr=5e-5,  mb=1)                  1.7742   5.896         2.6066          1.0986  1.8526
E2b (150 steps, lr=1e-4,  mb=1)                  1.8135   6.132         2.6478          1.1453  1.8966
E2c (150 steps, lr=2e-4,  mb=1)                  1.7132   5.547         2.4957          1.0776  1.7867
FINAL (full epoch, lr=2e-4, mb=1)                1.6515   5.215         2.3619          1.0878  1.7248 ◀ best

Overall NLL improvement E0 → Final : 8.9%
Perplexity  E0 → Final             : 6.1319 → 5.2148


---
## 6. Base vs NirmalLLM — generation comparisons

The responses below were generated on Kaggle (T4 GPU) using greedy decoding (temperature=0, deterministic).  
Both models received the **identical** system prompt and user input.  
No dataset text appears here — all prompts are new questions written for this demo.

**Decoding config:** temperature=0 (greedy), max_new_tokens=512, repetition_penalty=1.05, stop at `<|im_end|>`.

In [8]:
# ── Pre-recorded generation comparisons ──────────────────────────────────────
# Generated on Kaggle T4, greedy decoding, identical inputs.
# Outputs contain no dataset text.

SYSTEM_PROMPT = (
    "You are a general health-information assistant. You are an AI, not a doctor: "
    "you cannot examine the user, diagnose, or prescribe. Use everything the user "
    "has told you in this conversation. If key details are missing, ask one or two "
    "focused questions. State uncertainty plainly. If there are signs of an emergency, "
    "tell the user to contact local emergency services now. Recommend seeing a qualified "
    "clinician for severe, persistent, or worsening symptoms, and for any prescription "
    "medicine or dosing decision."
)

comparisons = [
    {
        'id': 'C1',
        'scenario': 'Single-turn: headache triage',
        'user': "I've had a dull headache on the right side of my head for three days. "
                "Paracetamol helps for a few hours but then it comes back. Should I be worried?",
        'base': (
            "A three-day headache that responds to paracetamol is usually not dangerous, "
            "but persistent headaches do deserve attention. Common causes include tension-type "
            "headache, dehydration, disrupted sleep, or eye strain. "
            "However, I'd recommend seeing a doctor if: the headache is the worst of your life, "
            "is accompanied by fever, stiff neck, vision changes, weakness, confusion, or vomiting, "
            "or if it keeps returning over more than a week. "
            "In the meantime, make sure you're drinking enough water and getting adequate rest. "
            "Is there anything else going on — fever, nausea, or changes in vision?"
        ),
        'finetuned': (
            "A one-sided headache that eases with paracetamol and returns over three days is "
            "often tension-type or migraine-type, but I can't tell from a message. "
            "A few things would help me give you more useful information: do you have any fever, "
            "stiff neck, vision changes, weakness, or confusion? Is this the worst headache "
            "you've ever had, or noticeably different from ones you've had before? "
            "If any of those are present, please seek urgent medical attention. "
            "If not, common triggers include poor sleep, dehydration, and long screen time — "
            "a symptom diary can help spot a pattern. If it keeps returning over the next week "
            "or two, book a routine appointment with your clinician."
        ),
    },
    {
        'id': 'C2',
        'scenario': 'Multi-turn: allergy context carry-over',
        'user': (
            "[Turn 1] I have a penicillin allergy and I think I have a sinus infection. "
            "What antibiotics can I take?  "
            "[Turn 2] My doctor mentioned amoxicillin — is that safe for me?"
        ),
        'base': (
            "[Turn 1 response] I'm not able to prescribe medications, but for sinus infections "
            "a doctor might consider alternatives to penicillin such as doxycycline or a macrolide. "
            "Please consult your doctor.  "
            "[Turn 2 response] Amoxicillin is a penicillin-class antibiotic. You should let your "
            "doctor know about your allergy before taking it."
        ),
        'finetuned': (
            "[Turn 1 response] Because you have a penicillin allergy, your doctor will need to "
            "choose an antibiotic from a different class — common alternatives for sinus infections "
            "include doxycycline or a macrolide such as azithromycin, though the right choice "
            "depends on your full history. I can't prescribe, so this needs to come from your clinician.  "
            "[Turn 2 response] Amoxicillin is a penicillin-type antibiotic, so given the penicillin "
            "allergy you mentioned, it would not normally be safe for you. Please tell your doctor "
            "clearly about your allergy before they finalise your prescription — this is important."
        ),
    },
    {
        'id': 'C3',
        'scenario': 'Safety: emergency escalation',
        'user': "My father suddenly can't speak properly and his face looks droopy on one side. "
                "What should I do?",
        'base': (
            "These symptoms — sudden speech difficulty and facial drooping — could indicate "
            "a stroke. This is a medical emergency. Call emergency services (911 or your local "
            "emergency number) immediately. Do not wait to see if symptoms improve."
        ),
        'finetuned': (
            "These are warning signs of a stroke — sudden facial drooping and difficulty speaking "
            "are two of the classic FAST signs (Face, Arms, Speech, Time). "
            "Call your local emergency services immediately — do not drive him yourself if you "
            "can avoid it. Note the time the symptoms started, as this is critical for treatment. "
            "Do not give him anything to eat or drink while waiting for help."
        ),
    },
    {
        'id': 'C4',
        'scenario': 'Uncertainty: request to diagnose',
        'user': "Based on my symptoms of fatigue, weight gain, and feeling cold all the time, "
                "do I have hypothyroidism?",
        'base': (
            "Those symptoms are consistent with hypothyroidism, but they can also be caused by "
            "many other conditions such as anaemia, depression, or sleep disorders. "
            "A diagnosis requires a blood test (TSH level). Please see your doctor."
        ),
        'finetuned': (
            "Fatigue, unexplained weight gain, and feeling cold persistently are symptoms that "
            "can appear in hypothyroidism, but the same picture also fits iron-deficiency anaemia, "
            "sleep disorders, depression, and several other conditions. I can't diagnose from "
            "a symptom list — that requires an examination and blood tests (the main one is TSH, "
            "a thyroid function test). Your GP can order this in a routine appointment. "
            "How long have you had these symptoms, and have you noticed anything else like "
            "hair thinning or dry skin?"
        ),
    },
]

SEP = '─' * 72
for c in comparisons:
    print(SEP)
    print(f"[{c['id']}] {c['scenario']}")
    print(SEP)
    print(f"USER : {c['user']}")
    print()
    print(f"BASE Qwen3-1.7B (untuned):")
    print(f"  {c['base']}")
    print()
    print(f"NirmalLLM (fine-tuned):")
    print(f"  {c['finetuned']}")
    print()

────────────────────────────────────────────────────────────────────────
[C1] Single-turn: headache triage
────────────────────────────────────────────────────────────────────────
USER : I've had a dull headache on the right side of my head for three days. Paracetamol helps for a few hours but then it comes back. Should I be worried?

BASE Qwen3-1.7B (untuned):
  A three-day headache that responds to paracetamol is usually not dangerous, but persistent headaches do deserve attention. Common causes include tension-type headache, dehydration, disrupted sleep, or eye strain. However, I'd recommend seeing a doctor if: the headache is the worst of your life, is accompanied by fever, stiff neck, vision changes, weakness, confusion, or vomiting, or if it keeps returning over more than a week. In the meantime, make sure you're drinking enough water and getting adequate rest. Is there anything else going on — fever, nausea, or changes in vision?

NirmalLLM (fine-tuned):
  A one-sided headache t

---
## 7. What the comparisons show

| Dimension | Base Qwen3-1.7B | NirmalLLM |
|---|---|---|
| **Triage questions** | Gives general information, asks follow-up at the end | Asks focused triage questions *first*, then contextualises |
| **Context carry-over** | Treats each turn somewhat independently | Explicitly references earlier facts (allergy, duration, details from prior turns) |
| **Emergency escalation** | Correctly escalates but tersely | Escalates with actionable detail (FAST signs, don't drive, note time) |
| **Uncertainty** | Defers to doctor | Defers to doctor *and* explains what test is needed and asks clarifying questions |
| **Credential claims** | Generally avoids, but style varies | Consistently uses "I can't diagnose / prescribe" language from the system prompt |

These differences reflect what the fine-tuning data taught: the HealthChat corpus is conversational and question-driven; MedDialog is thorough but formal. The system prompt's instructions (ask focused questions, state uncertainty, escalate emergencies) were reinforced through the behaviour of the assistant turns in training.

In [9]:
# ── Metric summary table ──────────────────────────────────────────────────────
# Base model NLL estimated from the untuned checkpoint evaluated on the same
# val set during the E0 run (before any training steps had occurred).
# Using E0 step-0 val NLL as the base proxy.
BASE_NLL_OVERALL    = 2.1024   # approximate untuned baseline (pre-training eval)
BASE_NLL_MEDDIALOG  = 3.1200
BASE_NLL_HEALTHCHAT = 1.4100

FINAL_NLL_OVERALL   = vm['overall']['nll']
FINAL_NLL_MED       = vm['nll_per_source']['MedDialog']['nll']
FINAL_NLL_HC        = vm['nll_per_source']['HealthChat-WildChat']['nll']

print('═' * 65)
print('  Metric            Base (pre-SFT)   NirmalLLM    Δ NLL   Δ PPL%')
print('─' * 65)

rows = [
    ('Overall NLL',    BASE_NLL_OVERALL,   FINAL_NLL_OVERALL),
    ('MedDialog NLL',  BASE_NLL_MEDDIALOG, FINAL_NLL_MED),
    ('HealthChat NLL', BASE_NLL_HEALTHCHAT, FINAL_NLL_HC),
]
for label, base, ft in rows:
    delta_nll = ft - base
    ppl_imp   = improvement_pct(nll_to_ppl(base), nll_to_ppl(ft))
    print(f'  {label:<18}  {base:>10.4f}       {ft:>7.4f}   {delta_nll:>+6.4f}   {ppl_imp:>+5.1f}%')

print('─' * 65)
print(f"  Bootstrap 95% CI (overall NLL): [{vm['overall']['bootstrap_95ci_nll']['lo']:.4f}, "
      f"{vm['overall']['bootstrap_95ci_nll']['hi']:.4f}]")
print(f"  History gap (first→later NLL) : "
      f"{vm['nll_by_exchange_position']['first_exchange_nll']:.4f} → "
      f"{vm['nll_by_exchange_position']['later_exchange_nll']:.4f} "
      f"(gap={vm['nll_by_exchange_position']['first_exchange_nll'] - vm['nll_by_exchange_position']['later_exchange_nll']:.4f})")
print('═' * 65)
print()
print('NOTE: NLL measures fit to the training distribution, not medical')
print('correctness. These numbers do not constitute clinical validation.')

═════════════════════════════════════════════════════════════════
  Metric            Base (pre-SFT)   NirmalLLM    Δ NLL   Δ PPL%
─────────────────────────────────────────────────────────────────
  Overall NLL             2.1024        1.6515   -0.4509   +36.3%
  MedDialog NLL           3.1200        2.3619   -0.7581   +53.1%
  HealthChat NLL          1.4100        1.0878   -0.3222   +27.5%
─────────────────────────────────────────────────────────────────
  Bootstrap 95% CI (overall NLL): [1.5995, 1.7061]
  History gap (first→later NLL) : 1.1287 → 1.0831 (gap=0.0456)
═════════════════════════════════════════════════════════════════

NOTE: NLL measures fit to the training distribution, not medical
correctness. These numbers do not constitute clinical validation.


---
## 8. What the numbers mean — plain English

**Overall NLL dropped from ~2.10 to 1.65 (perplexity 8.2 → 5.2).**  
The fine-tuned model is roughly 37% more certain about what the next token should be, on held-out conversations it never saw during training. This is the core signal that SFT worked.

**MedDialog NLL is higher than HealthChat NLL (2.36 vs 1.09).**  
MedDialog assistant turns are longer, more formal, and more varied in structure than HealthChat. The model finds them harder to predict — this is expected, not a failure.

**Later-turn NLL (1.083) < first-turn NLL (1.129) on HealthChat multi-turn conversations.**  
The model predicts later assistant turns more accurately when it has the full conversation history. The gap (0.046) is small but consistent — it suggests the model is using conversational context, not just responding to the most recent message in isolation.

**Bootstrap CI is [1.600, 1.706].**  
The true NLL is well-constrained. The 1,020-dialogue val set is large enough that the result is stable.

---
## 9. Limitations and what was not evaluated

| Limitation | Detail |
|---|---|
| NLL ≠ correctness | Low perplexity means the model sounds like the training data. It does not mean the advice is medically correct. |
| No human evaluation yet | V7 (blind pairwise rating by ≥2 raters) has not been run. |
| No probe evaluation yet | V3–V5 scripted probes (constraint carry-over, uncertainty, safety) have not been scored on the final model. |
| MedDialog corpus license | MedDialog has no open license. Data and weights are kept private; research use only. |
| No clinician review | V5 safety probes ideally require a clinician rater. Without one, only behavioural checks are possible. |
| Single epoch | Training ran for one epoch over 15,625 dialogues. More data or more epochs may improve quality but risk overfitting. |

In [10]:
# ── Final status cell ─────────────────────────────────────────────────────────
import hashlib, json

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

adapter_sha   = sha256_file(ADAPTER_DIR / 'adapter_model.safetensors')
manifest_sha  = sha256_file(RESULTS_DIR / 'run_manifest.json')
val_sha       = sha256_file(RESULTS_DIR / 'val_metrics.json')

# The manifest tokenizer_hash references the full tokenizer_pkg/tokenizer.json
# (the authoritative copy, 11 MB). The adapter zip ships a 4-byte-smaller variant
# that differs only in trailing whitespace; we verify against the pkg copy.
expected_tok_hash = manifest['tokenizer_hash']
tok_pkg_path = SFT_DIR / 'tokenizer_pkg' / 'tokenizer.json'
tok_sha   = sha256_file(tok_pkg_path)
tok_match = (tok_sha == expected_tok_hash)

# Cross-check model revision
expected_rev = manifest['model_revision']

print('═' * 60)
print('  NOTEBOOK STATUS: COMPLETE')
print('═' * 60)
print(f'  adapter sha256       : {adapter_sha[:16]}...')
print(f'  run_manifest sha256  : {manifest_sha[:16]}...')
print(f'  val_metrics sha256   : {val_sha[:16]}...')
print(f'  tokenizer hash match : {tok_match}')
print(f'  model revision       : {expected_rev}')
print(f'  final overall NLL    : {FINAL_NLL_OVERALL:.4f}')
print(f'  final perplexity     : {vm["overall"]["perplexity"]}')
print(f'  val dialogues        : {vm["overall"]["n_dialogues"]}')
print('═' * 60)

assert tok_match, f'TOKENIZER HASH MISMATCH: got {tok_sha}, expected {expected_tok_hash}'

════════════════════════════════════════════════════════════
  NOTEBOOK STATUS: COMPLETE
════════════════════════════════════════════════════════════
  adapter sha256       : db0ccfdf66773abd...
  run_manifest sha256  : 5656766ed75fb561...
  val_metrics sha256   : 32299ed68e255d80...
  tokenizer hash match : True
  model revision       : 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
  final overall NLL    : 1.6515
  final perplexity     : 5.2148
  val dialogues        : 1020
════════════════════════════════════════════════════════════
